# 05 — Simulation basics

A first discrete-event simulation with SimPy, on a toy example:
one machine, lots arriving every 30 min, Gamma processing times (mean 50 min).

The machine is too slow for the arrival rate, so a queue should build up.

In [5]:
import numpy as np
import pandas as pd
import simpy

rng = np.random.default_rng(0)


def lot(env, name, machine, log):
    arrive = env.now
    print(f"{env.now:6.1f}  {name} arrives")

    with machine.request() as req:
        yield req  # wait until the machine is free
        start = env.now
        print(f"{env.now:6.1f}  {name} starts (waited {start - arrive:.1f} min)")
        yield env.timeout(rng.gamma(2.5, 20))  # processing

    print(f"{env.now:6.1f}  {name} leaves")
    log.append({"lot": name, "arrive": arrive, "start": start, "end": env.now})


def arrivals(env, machine, log):
    for n in range(10):
        env.process(lot(env, f"LOT_{n}", machine, log))
        yield env.timeout(30)  # next lot in 30 min


env = simpy.Environment()
machine = simpy.Resource(env, capacity=1)
log = []

env.process(arrivals(env, machine, log))
env.run()

   0.0  LOT_0 arrives
   0.0  LOT_0 starts (waited 0.0 min)
  30.0  LOT_1 arrives
  47.1  LOT_0 leaves
  47.1  LOT_1 starts (waited 17.1 min)
  60.0  LOT_2 arrives
  90.0  LOT_3 arrives
 112.2  LOT_1 leaves
 112.2  LOT_2 starts (waited 52.2 min)
 120.0  LOT_4 arrives
 141.6  LOT_2 leaves
 141.6  LOT_3 starts (waited 51.6 min)
 150.0  LOT_5 arrives
 180.0  LOT_6 arrives
 210.0  LOT_7 arrives
 235.8  LOT_3 leaves
 235.8  LOT_4 starts (waited 115.8 min)
 240.0  LOT_8 arrives
 261.5  LOT_4 leaves
 261.5  LOT_5 starts (waited 111.5 min)
 270.0  LOT_9 arrives
 289.0  LOT_5 leaves
 289.0  LOT_6 starts (waited 109.0 min)
 293.6  LOT_6 leaves
 293.6  LOT_7 starts (waited 83.6 min)
 309.6  LOT_7 leaves
 309.6  LOT_8 starts (waited 69.6 min)
 338.8  LOT_8 leaves
 338.8  LOT_9 starts (waited 68.8 min)
 395.4  LOT_9 leaves


In [6]:
# Results
res = pd.DataFrame(log)
res["wait"] = res["start"] - res["arrive"]
res.round(1)

,lot,arrive,start,end,wait
0,LOT_0,0,0.0,47.1,0.0
1,LOT_1,30,47.1,112.2,17.1
2,LOT_2,60,112.2,141.6,52.2
3,LOT_3,90,141.6,235.8,51.6
4,LOT_4,120,235.8,261.5,115.8
5,LOT_5,150,261.5,289.0,111.5
6,LOT_6,180,289.0,293.6,109.0
7,LOT_7,210,293.6,309.6,83.6
8,LOT_8,240,309.6,338.8,69.6
9,LOT_9,270,338.8,395.4,68.8


## Custom dispatching

SimPy resources serve lots in FIFO order. Our rule is different:
pick a waiting lot at random, with probability proportional to its product score.
So machines are written as processes that pick from a shared waiting list themselves.

Toy check: products A (score 1) and B (score 0.25), one machine busy ~90% of the time.

B should wait longer than A.

In [7]:
# Workshop with product-score dispatching

class Workshop:
    def __init__(self, env, n_machines, scores, rng):
        self.env = env
        self.scores = scores
        self.rng = rng
        self.queue = []
        self.log = []
        self.new_lot = env.event()  # signal to wake idle machines

        for m in range(n_machines):
            env.process(self.machine(f"M{m}"))

    def arrive(self, lot):
        lot["arrive"] = self.env.now
        self.queue.append(lot)
        self.new_lot.succeed()
        self.new_lot = self.env.event()  # fresh signal for the next arrival

    def pick(self):
        # random pick, weighted by product score
        w = np.array([self.scores[lot["product"]] for lot in self.queue])
        i = self.rng.choice(len(self.queue), p=w / w.sum())
        return self.queue.pop(i)

    def machine(self, name):
        while True:
            if not self.queue:
                yield self.new_lot  # idle until a lot shows up
                continue
            lot = self.pick()
            lot["machine"] = name
            lot["start"] = self.env.now
            yield self.env.timeout(self.rng.gamma(2.5, 11))  # mean 27.5 min
            lot["end"] = self.env.now
            self.log.append(lot)


def arrivals(env, ws, n_lots, rng):
    for n in range(n_lots):
        ws.arrive({"lot": n, "product": rng.choice(["A", "B"])})
        yield env.timeout(30)

In [10]:
# Run and compare waits per product

rng = np.random.default_rng(0)
env = simpy.Environment()
ws = Workshop(env, n_machines=1, scores={"A": 1.0, "B": 0.25}, rng=rng)

env.process(arrivals(env, ws, n_lots=2000, rng=rng))
env.run()

res = pd.DataFrame(ws.log)
res["wait"] = res["start"] - res["arrive"]
print(f"machine busy: {(res['end'] - res['start']).sum() / res['end'].max():.1%}")
res.groupby("product")["wait"].agg(["count", "mean", "median"]).round(1)

machine busy: 90.3%


,count,mean,median
product,,,
A,993,34.5,13.4
B,1007,65.5,16.1


# 

### Takeaways :

- With scores A = 1, B = 0.25: B waits ~2× longer than A (65.5 vs 34.5 min)
- With equal scores: 53.9 vs 46.3 min, no systematic gap (the difference is random noise over 2,000 lots)
- Mean wait over all lots is ~50 min in both cases: priority does not reduce total waiting,
  it redistributes it between products
- One run is noisy: the twin will be run several times with different seeds (replications)